# Table Lineage — Test Notebook
Tests `table_lineage_er_tab.py` logic end-to-end in Jupyter.

**Routing logic:**
- Table ends with `_profile_curr_ikg` → queries pre-built `ikg_lineage_<table>`
- Any other table → queries `ikg_table_lineage_metadata_auto_refresh`  
  shows **upstream** (sources) + **downstream** (consumers) as two sections

**Steps:** Fill in DB config → `Kernel → Restart & Run All` → select table → click **Show Lineage ▶**

In [ ]:
# ── DB Config — edit these ────────────────────────────────────────────
GP_HOST    = 'greenplum-rdsp.zur.swissbank.com'
GP_PORT    = 5432
GP_DB      = 'gprdsp'
GP_USER    = 'ds_rdsp_dev'
IKG_SCHEMA = 'sandbox_prj_smart_insights'

# GitLab base for SQL file links (matches ikg_lineage_sql_stitcher.py)
GITLAB_BASE = (
    'https://devcloud.ubs.net/ubs/gwma/smart-technology-and-analytics/'
    'staat-data-science/staat-ds-genesis/genesis-platform/'
    'ikg-dags/-/blob/ikg-master/'
)

import json, re, tempfile
from pathlib import Path
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, IFrame
from sqlalchemy import create_engine, text
import getpass

_pw = getpass.getpass(f'Greenplum password for {GP_USER}@{GP_HOST}:')
_engine = create_engine(
    f'postgresql+psycopg2://{GP_USER}:{_pw}@{GP_HOST}:{GP_PORT}/{GP_DB}',
    pool_pre_ping=True,
)
def _run(sql):
    with _engine.connect() as conn:
        return pd.read_sql(text(sql), conn)
print('DB ready')


In [ ]:
# ── Data functions (mirrors utils/data.py table lineage section) ───────

def is_profile_table(table_name):
    """True if table name ends with _profile_curr_ikg."""
    return str(table_name).lower().strip().endswith('_profile_curr_ikg')

def get_all_lineage_tables():
    """All distinct table names from the master lineage table."""
    return _run(f'''
        SELECT DISTINCT table_name FROM (
            SELECT TRIM(target_table) AS table_name
            FROM {IKG_SCHEMA}.ikg_table_lineage_metadata_auto_refresh
            WHERE target_table IS NOT NULL AND TRIM(target_table) <> ''
            UNION
            SELECT TRIM(source_table)
            FROM {IKG_SCHEMA}.ikg_table_lineage_metadata_auto_refresh
            WHERE source_table IS NOT NULL AND TRIM(source_table) <> ''
        ) t ORDER BY table_name''')

def get_table_lineage_for_profile(table_name):
    """Profile tables: query ikg_lineage_<table_name>."""
    safe = str(table_name).lower().strip().replace("'", "''")
    return _run(f'''
        SELECT order_index, root_profile_table, target_table,
               source_schema, source_table, process, filename, filepath
        FROM {IKG_SCHEMA}.ikg_lineage_{safe}
        WHERE source_table IS NOT NULL AND TRIM(source_table) <> ''
        ORDER BY order_index ASC''')

def get_table_lineage_upstream(table_name):
    """Upstream lineage via single WITH RECURSIVE on master table (Greenplum safe)."""
    safe = str(table_name).replace("'", "''")
    return _run(f'''
        WITH RECURSIVE upstream AS (
            SELECT target_table, source_table, source_schema,
                   process, filename, filepath, 0 AS depth
            FROM {IKG_SCHEMA}.ikg_table_lineage_metadata_auto_refresh
            WHERE LOWER(TRIM(target_table)) = LOWER(TRIM(\'{safe}\'))
              AND source_table IS NOT NULL AND TRIM(source_table) <> ''
            UNION ALL
            SELECT m.target_table, m.source_table, m.source_schema,
                   m.process, m.filename, m.filepath, u.depth + 1
            FROM {IKG_SCHEMA}.ikg_table_lineage_metadata_auto_refresh m
            JOIN upstream u
              ON LOWER(TRIM(m.target_table)) = LOWER(TRIM(u.source_table))
            WHERE u.depth < 6
              AND m.source_table IS NOT NULL AND TRIM(m.source_table) <> ''
        ) SELECT DISTINCT * FROM upstream''')

def get_table_lineage_downstream(table_name):
    """Downstream lineage via single WITH RECURSIVE on master table (Greenplum safe)."""
    safe = str(table_name).replace("'", "''")
    return _run(f'''
        WITH RECURSIVE downstream AS (
            SELECT target_table, source_table, source_schema,
                   process, filename, filepath, 0 AS depth
            FROM {IKG_SCHEMA}.ikg_table_lineage_metadata_auto_refresh
            WHERE LOWER(TRIM(source_table)) = LOWER(TRIM(\'{safe}\'))
              AND target_table IS NOT NULL AND TRIM(target_table) <> ''
            UNION ALL
            SELECT m.target_table, m.source_table, m.source_schema,
                   m.process, m.filename, m.filepath, d.depth + 1
            FROM {IKG_SCHEMA}.ikg_table_lineage_metadata_auto_refresh m
            JOIN downstream d
              ON LOWER(TRIM(m.source_table)) = LOWER(TRIM(d.target_table))
            WHERE d.depth < 4
              AND m.target_table IS NOT NULL AND TRIM(m.target_table) <> ''
        ) SELECT DISTINCT * FROM downstream''')

print('Data functions ready')


In [ ]:
# ── Schema colours (matches table_lineage_template.html RAW2COLOR) ──────
RAW2COLOR = {
    'ikg_schema':'#1976d2','sandbox_prj_smart_insights':'#1976d2',
    'sandbox_ikg_pre_prd':'#1976d2','edw_input_schema':'#00897b',
    'core_wma_shared':'#00897b','edw_view_input_schema':'#00897b',
    'ikg_vendor_schema':'#00897b','core_wma_shared_masked':'#26a69a',
    'sandbox_wma_shared':'#388e3c','sandbox_prj_ds_data':'#f4511e',
    'sandbox_prj_sbl':'#7b1fa2','core_nlg':'#5c6bc0','nlg_schema':'#5c6bc0',
    'core_model':'#6d4c41','model_schema':'#6d4c41',
    'sandbox_prj_dsforoverdrive':'#0288d1','core_in_shared':'#689f38',
    'ikg_clip_schema':'#689f38','sandbox_prj_adhoc':'#e91e8c',
    'ikg_wealthx_schema':'#00838f','sandbox_prj_rbat':'#9e7c0a',
    '_default':'#607d8b',
}
LEGEND_DATA = [
    {'label':'ikg',                             'color':'#1976d2'},
    {'label':'core_wma_shared / edw / source',  'color':'#00897b'},
    {'label':'core_nlg',                        'color':'#5c6bc0'},
    {'label':'core_model',                      'color':'#6d4c41'},
    {'label':'temp',                            'color':'#607d8b'},
]
print('Colours ready')


In [ ]:
# ── Graph builder → ALL_GRAPHS format (mirrors table_lineage_er_tab.py) ─

def _s(v):
    if v is None: return ''
    try:
        if pd.isna(v): return ''
    except Exception: pass
    s = str(v).strip()
    return '' if s.lower() in ('none','nan') else s

def _gitlab_url(filepath):
    fp = _s(filepath)
    if not fp: return ''
    return GITLAB_BASE + fp.lstrip('/')

def _rows_to_graph_section(df, focal_table, section_label):
    """
    Build one ALL_GRAPHS entry from a lineage DataFrame.
    Columns expected: target_table, source_table, source_schema,
                      process, filename, filepath.
    Node format: {id, tbl, col, schema, is_start, web_url}
    nrows:       {node_id: [{target_table, source_table, process,
                              sql_process, logic, ...}]}
    """
    if df.empty: return None
    nodes_dict = {}; edges_set = set(); edges_list = []; nrows = {}

    def _ensure(table, schema, filename, filepath, is_start=False):
        if table not in nodes_dict:
            nodes_dict[table] = {
                'id': table, 'tbl': table,
                'col': Path(filename).stem if filename else '',
                'schema': schema, 'is_start': is_start,
                'web_url': _gitlab_url(filepath),
            }
        else:
            nd = nodes_dict[table]
            if schema   and not nd['schema']:  nd['schema']  = schema
            if filename and not nd['col']:     nd['col']     = Path(filename).stem
            if filepath and not nd['web_url']: nd['web_url'] = _gitlab_url(filepath)

    _ensure(focal_table, '', '', '', is_start=True)
    nrows[focal_table] = []

    for r in df.itertuples(index=False):
        tgt   = _s(getattr(r,'target_table', None))
        src   = _s(getattr(r,'source_table', None))
        sch   = _s(getattr(r,'source_schema',None))
        proc  = _s(getattr(r,'process',      None))
        fname = _s(getattr(r,'filename',     None))
        fpath = _s(getattr(r,'filepath',     None))
        if not tgt or not src: continue
        _ensure(tgt, sch, fname, fpath, is_start=(tgt==focal_table))
        _ensure(src, sch, fname, fpath)
        ek = (src, tgt)
        if ek not in edges_set:
            edges_set.add(ek)
            edges_list.append({'from':src,'to':tgt})
        row_d = {
            'target_table': tgt,  'target_schema': sch,
            'sub_target_table': tgt, 'sub_target_schema': sch,
            'target_column': '',
            'source_table': src,  'source_schema': sch,
            'source_column': '',
            'process': proc,      'sql_process': 'sql',
            'logic': fname,       # SQL filename shown in monospace
        }
        nrows.setdefault(tgt, []).append(row_d)

    for nid in nodes_dict: nrows.setdefault(nid, [])
    if len(nodes_dict) <= 1: return None

    return {
        'profile_table': focal_table,
        'rule_column':   focal_table,
        'insight_type':  '',
        'section_label': section_label,
        'nodes':  list(nodes_dict.values()),
        'edges':  edges_list,
        'nrows':  nrows,
    }

def build_all_graphs(table_name):
    """Route to profile or general lineage and return ALL_GRAPHS list."""
    table_name = _s(table_name)
    if not table_name: return []

    if is_profile_table(table_name):
        print(f'Profile table detected → using ikg_lineage_{table_name.lower().strip()}')
        try:
            df = get_table_lineage_for_profile(table_name)
            print(f'  {len(df)} lineage rows loaded')
        except Exception as e:
            print(f'Error loading profile lineage: {e}'); return []
        sec = _rows_to_graph_section(df, table_name, 'upstream lineage')
        return [sec] if sec else []
    else:
        print(f'General table → querying ikg_table_lineage_metadata_auto_refresh')
        sections = []
        try:
            df_up = get_table_lineage_upstream(table_name)
            print(f'  Upstream rows: {len(df_up)}')
            if not df_up.empty:
                sec = _rows_to_graph_section(df_up, table_name, 'upstream (sources)')
                if sec: sections.append(sec)
        except Exception as e:
            print(f'  Upstream error: {e}')
        try:
            df_down = get_table_lineage_downstream(table_name)
            print(f'  Downstream rows: {len(df_down)}')
            if not df_down.empty:
                sec = _rows_to_graph_section(df_down, table_name, 'downstream (consumers)')
                if sec: sections.append(sec)
        except Exception as e:
            print(f'  Downstream error: {e}')
        return sections

print('Graph builder ready')


In [ ]:
# ── Template renderer ───────────────────────────────────────────────────
_SEARCH = [
    Path('utils/table_lineage_template.html'),
    Path('../utils/table_lineage_template.html'),
    Path('insights_metadata_dashboard/utils/table_lineage_template.html'),
]
TEMPLATE_PATH = next((p for p in _SEARCH if p.exists()), None)
if not TEMPLATE_PATH:
    raise FileNotFoundError(
        f'table_lineage_template.html not found. Searched: {_SEARCH}'
    )
print(f'Template: {TEMPLATE_PATH.resolve()}')

def render_html(table_name):
    template   = TEMPLATE_PATH.read_text(encoding='utf-8')
    all_graphs = build_all_graphs(table_name)
    if not all_graphs:
        return (
            f'<html><body style="font-family:Arial;padding:30px;">'
            f'<b>No lineage data found for: {table_name}</b></body></html>'
        )
    payload = (
        f'var ALL_GRAPHS  = {json.dumps(all_graphs)};\n'
        f'var RAW2COLOR   = {json.dumps(RAW2COLOR)};\n'
        f'var LEGEND_DATA = {json.dumps(LEGEND_DATA)};'
    )
    # lambda prevents re.sub interpreting backslashes in JSON payload
    out = re.sub(
        r'var ALL_GRAPHS\s*=\s*.*?;\s*var RAW2COLOR\s*=\s*.*?;\s*var LEGEND_DATA\s*=\s*.*?;',
        lambda _: payload,
        template,
        flags=re.DOTALL,
    )
    out = re.sub(
        r'<title>.*?</title>',
        f'<title>{table_name} — Table Lineage</title>',
        out, count=1, flags=re.DOTALL,
    )
    return out

print('Renderer ready')


In [ ]:
# ── Load table list ─────────────────────────────────────────────────────
print('Loading tables from ikg_table_lineage_metadata_auto_refresh ...')
try:
    _df_tables = get_all_lineage_tables()
    _table_list = _df_tables['table_name'].dropna().astype(str).tolist()
    print(f'{len(_table_list)} tables loaded')
except Exception as e:
    print(f'Warning: could not load table list ({e}). You can still type a table name.')
    _table_list = []

# Quick summary
profile_count = sum(1 for t in _table_list if t.lower().endswith('_profile_curr_ikg'))
print(f'  Profile tables (_profile_curr_ikg): {profile_count}')
print(f'  Other tables: {len(_table_list) - profile_count}')


In [ ]:
# ── Interactive widgets ──────────────────────────────────────────────────
_ly = widgets.Layout(width='560px')
_st = {'description_width': '110px'}

w_table = widgets.Combobox(
    options    = _table_list,
    value      = '',
    placeholder= 'Type to search or select a table ...',
    description= 'Table:',
    ensure_option= False,   # allow typing any name not in the list
    layout     = _ly,
    style      = _st,
)
w_type = widgets.Label(
    value= '',
    style= {'description_width': '0px'},
)
w_btn = widgets.Button(
    description = 'Show Lineage ▶',
    button_style= 'success',
    layout      = widgets.Layout(width='160px', margin='8px 0 0 0'),
)
w_out = widgets.Output()

def _on_table(change):
    val = change['new']
    if not val:
        w_type.value = ''
        return
    if is_profile_table(val):
        w_type.value = f'  ✅ Profile table → will use ikg_lineage_{val.lower().strip()}'
    else:
        w_type.value = '  🔍 General table → upstream + downstream from master table'

def _on_btn(b):
    w_out.clear_output()
    table = w_table.value.strip()
    if not table:
        with w_out: print('⚠  Enter or select a table name.')
        return
    with w_out:
        print(f'Building lineage for: {table} ...')
    try:
        html_content = render_html(table)
    except Exception as exc:
        with w_out:
            w_out.clear_output(wait=True)
            print(f'❌  Error: {exc}')
        return
    tmp = Path(tempfile.mktemp(suffix='.html', dir='.'))
    tmp.write_text(html_content, encoding='utf-8')
    with w_out:
        w_out.clear_output(wait=True)
        display(IFrame(src=str(tmp), width='100%', height='760px'))

w_table.observe(_on_table, names='value')
w_btn.on_click(_on_btn)

display(widgets.VBox([
    widgets.HTML('<b>Table Lineage Explorer</b>'),
    widgets.HBox([w_table]),
    w_type,
    w_btn,
    w_out,
]))


In [ ]:
# ── Cell 9: Headless test (optional — no widgets) ───────────────────────
# Uncomment one of the lines below to test directly without the widget UI.

# -- Profile table example --
# TABLE_TO_TEST = 'account_profile_curr_ikg'

# -- General (non-profile) table example --
# TABLE_TO_TEST = 'ubs_base_acc_features_ikg'

# graphs = build_all_graphs(TABLE_TO_TEST)
# for g in graphs:
#     print(f"Section: {g['section_label']}")
#     print(f"  Nodes: {len(g['nodes'])}  Edges: {len(g['edges'])}")
#     print(f"  First 3 nodes: {[n['id'] for n in g['nodes'][:3]]}")

# -- Render and save HTML --
# html_content = render_html(TABLE_TO_TEST)
# out_path = Path(f'{TABLE_TO_TEST}_lineage.html')
# out_path.write_text(html_content, encoding='utf-8')
# print(f'Saved: {out_path.resolve()}')
# display(IFrame(src=str(out_path), width='100%', height='760px'))

print('Cell 9 ready — uncomment lines above to run a headless test')
